# VISIONSTREAM AI — SPRINT 2 / TASK S2.4
## GATE 2: Full 50-Epoch Training Execution & Checkpoint Export

**Project:** VisionStream AI  
**Phase:** Phase 3 (Operational Sprint 2 — Model Training & Evaluation)  
**Objective:** Execute full 50-epoch fine-tuning on Google Colab Free GPU, capture same-session GPU/CUDA evidence, and export checkpoints (`best.pt`, `last.pt`) and full metrics to Google Drive.  
**Strict Boundary:** USD 0 cloud expenditure. Sealed Holdout Test. Evidence-driven engineering.

### Step 1: Capture Same-Session GPU & Runtime Diagnostics

In [ ]:
import sys, subprocess, os
from datetime import datetime, timezone
from pathlib import Path

SESSION_LOG_DIR = Path('/content/runs/gate2_diagnostics')
SESSION_LOG_DIR.mkdir(parents=True, exist_ok=True)
diag_log_path = SESSION_LOG_DIR / 'session_gpu_diagnostic.txt'

diag_lines = []
diag_lines.append("=" * 70)
diag_lines.append("VISIONSTREAM AI — S2.4 GATE 2 SAME-SESSION GPU/CUDA DIAGNOSTIC")
diag_lines.append("=" * 70)

start_utc = datetime.now(timezone.utc).isoformat()
diag_lines.append(f"Start Timestamp (UTC): {start_utc}")
diag_lines.append(f"Python Version       : {sys.version.split()[0]}")

import torch, torchvision
diag_lines.append(f"PyTorch Version      : {torch.__version__}")
diag_lines.append(f"Torchvision Version  : {torchvision.__version__}")
cuda_ok = torch.cuda.is_available()
diag_lines.append(f"CUDA Available       : {cuda_ok}")

if not cuda_ok:
    raise SystemExit("FATAL: CUDA is NOT available! Stop and report BLOCKED. Do not train on CPU.")

dev_count = torch.cuda.device_count()
diag_lines.append(f"CUDA Device Count    : {dev_count}")
for i in range(dev_count):
    dev_name = torch.cuda.get_device_name(i)
    vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    cap = torch.cuda.get_device_capability(i)
    diag_lines.append(f"  - Device [{i}]: {dev_name}")
    diag_lines.append(f"    Total VRAM        : {vram_gb:.2f} GB")
    diag_lines.append(f"    Compute Capability: {cap[0]}.{cap[1]}")
diag_lines.append(f"PyTorch Built with CUDA: {torch.version.cuda}")
diag_lines.append(f"cuDNN Version          : {torch.backends.cudnn.version()}")

diag_lines.append("\n" + "-" * 70)
diag_lines.append("NVIDIA SYSTEM MANAGEMENT INTERFACE (nvidia-smi)")
diag_lines.append("-" * 70)
res = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False)
diag_lines.append(res.stdout if res.returncode == 0 else res.stderr)
diag_lines.append("=" * 70)

diag_text = "\n".join(diag_lines)
print(diag_text)
diag_log_path.write_text(diag_text)
print(f"\nSame-session diagnostic saved to: {diag_log_path}")

### Step 2: Install Ultralytics and Record Version

In [ ]:
!pip install ultralytics
import ultralytics
print(f"Ultralytics Version: {ultralytics.__version__}")

### Step 3: Mount Google Drive & Verify Dataset Archive Checksum

In [ ]:
from google.colab import drive
import hashlib, zipfile, shutil
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ZIP = Path('/content/drive/MyDrive/coco-person-sprint2.zip')
if not DRIVE_ZIP.exists():
    matches = list(Path('/content/drive/MyDrive').rglob('coco-person-sprint2.zip'))
    if matches:
        DRIVE_ZIP = matches[0]
    else:
        raise FileNotFoundError("Archive not found in Drive! Please upload coco-person-sprint2.zip to MyDrive")

print(f"Found archive at: {DRIVE_ZIP}")
archive_bytes = DRIVE_ZIP.read_bytes()
archive_sha = hashlib.sha256(archive_bytes).hexdigest()
print(f"Drive Archive Size: {len(archive_bytes)} bytes")
print(f"Drive Archive SHA256: {archive_sha}")

EXPECTED_ZIP_SHA = "ad28fb9e221d9a781cdaf88334beb93897db9898c20df0125a035aa59fa0de9e"
if archive_sha != EXPECTED_ZIP_SHA:
    raise ValueError(f"FATAL: Archive SHA256 mismatch! Expected {EXPECTED_ZIP_SHA}, got {archive_sha}")
print("Archive SHA256 MATCH VERIFIED! (100% Identical)")

### Step 4: Extract Dataset & Verify Per-Image Checksums (Sealed Test Protection)

In [ ]:
EXTRACT_DIR = Path('/content/dataset/coco-person')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting dataset to ephemeral storage...")
with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

train_imgs = list((EXTRACT_DIR / 'images/train').glob('*.jpg'))
val_imgs = list((EXTRACT_DIR / 'images/val').glob('*.jpg'))
test_imgs = list((EXTRACT_DIR / 'images/test').glob('*.jpg'))
print(f"Extracted: Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}")
assert len(train_imgs) == 2000, f"Expected 2000 train, got {len(train_imgs)}"
assert len(val_imgs) == 239, f"Expected 239 val, got {len(val_imgs)}"
assert len(test_imgs) == 0, "FATAL: Holdout test leaked into training directory!"

checksum_file = EXTRACT_DIR / 'metadata/checksums.sha256'
expected_hashes = {}
for line in checksum_file.read_text().splitlines():
    if line.strip():
        sha, rel = line.strip().split('  ')
        expected_hashes[rel] = sha

mismatches = 0
checked = 0
for rel, exp_sha in expected_hashes.items():
    if 'test' in rel:
        continue # test split is strictly sealed at local host
    img_path = EXTRACT_DIR / rel
    if not img_path.exists():
        mismatches += 1
        continue
    act_sha = hashlib.sha256(img_path.read_bytes()).hexdigest()
    if act_sha != exp_sha:
        mismatches += 1
    checked += 1

print(f"Checked {checked} training & validation images against canonical SHA-256: {mismatches} mismatches.")
if mismatches > 0:
    raise ValueError(f"FATAL: Checksum verification failed with {mismatches} mismatches! ABORTING.")
print("PER-IMAGE SHA-256 INTEGRITY VERIFIED (100% MATCH)!")

### Step 5: Configure colab_dataset.yaml

In [ ]:
yaml_content = f"""# Ultralytics YOLOv8 Dataset Configuration - Colab Full Training
path: {EXTRACT_DIR}
train: images/train
val: images/val

names:
  0: person
"""
colab_yaml = EXTRACT_DIR / 'colab_dataset.yaml'
colab_yaml.write_text(yaml_content)
print("colab_dataset.yaml created:")
print(colab_yaml.read_text())

### Step 6: Verify Baseline Checkpoint & Audit Weight Initialization

In [ ]:
from ultralytics import YOLO

DRIVE_MODEL = Path('/content/drive/MyDrive/yolov8n.pt')
if DRIVE_MODEL.exists():
    shutil.copy(DRIVE_MODEL, 'yolov8n.pt')
    print("Copied verified baseline yolov8n.pt from Drive.")

model = YOLO('yolov8n.pt')
model_path = Path('yolov8n.pt')
if model_path.exists():
    m_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()
    print(f"Starting yolov8n.pt SHA256: {m_sha}")
    EXPECTED_MODEL_SHA = "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36"
    if m_sha == EXPECTED_MODEL_SHA:
        print("Starting model SHA-256 matches exact project baseline!")
    else:
        print(f"Notice: Starting model SHA ({m_sha}) differs from local commit ({EXPECTED_MODEL_SHA})")

### Step 7: Execute Full 50-Epoch Training Run (Authorized Gate 2)

In [ ]:
import time

print("=== STARTING GATE 2: FULL 50-EPOCH TRAINING RUN ===")
start_train_time = time.time()
start_train_utc = datetime.now(timezone.utc).isoformat()
print(f"Training Start Time (UTC): {start_train_utc}")

results = model.train(
    data=str(colab_yaml),
    epochs=50,              # FULL TRAINING RUN
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=10,            # Early stopping patience 10
    seed=42,
    amp=True,
    workers=4,
    close_mosaic=10,
    dropout=0.0,
    save=True,
    save_period=-1,
    project='/content/runs/gate2_training',
    name='full_run',
    exist_ok=True,
    plots=True
)

end_train_time = time.time()
end_train_utc = datetime.now(timezone.utc).isoformat()
elapsed_train = end_train_time - start_train_time
print(f"=== TRAINING FINISHED AT {end_train_utc} ===")
print(f"Total Duration: {elapsed_train:.2f} seconds ({elapsed_train / 60:.2f} minutes)")

### Step 8: Post-Training GPU Memory Inspection & Export Artifacts to Drive

In [ ]:
print("-" * 70)
print("POST-TRAINING GPU STATUS (nvidia-smi)")
print("-" * 70)
!nvidia-smi

import pandas as pd

run_dir = Path('/content/runs/gate2_training/full_run')
csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"\nTraining completed epochs: {len(df)}")
    print("\nFinal 5 Epochs Summary:")
    print(df.tail(5).to_string())

# Copy session diagnostic into run_dir
if diag_log_path.exists():
    shutil.copy(diag_log_path, run_dir / 'session_gpu_diagnostic.txt')

# Export artifacts to Google Drive
DEST_EVIDENCE = Path('/content/drive/MyDrive/visionstream_gate2_evidence')
DEST_EVIDENCE.mkdir(parents=True, exist_ok=True)

print(f"\nExporting all artifacts from {run_dir} to {DEST_EVIDENCE}...")
for f in run_dir.rglob('*'):
    if f.is_file():
        rel = f.relative_to(run_dir)
        target = DEST_EVIDENCE / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, target)
        print(f"  -> Exported: {rel} ({f.stat().st_size} bytes)")

print("\nALL ARTIFACTS SUCCESSFULLY EXPORTED TO GOOGLE DRIVE!")